In [1]:
from sklearn.datasets import  load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import  LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd 

In [2]:
data = load_breast_cancer(as_frame= True)
df = data.frame

In [3]:
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
train_df, temp_df = train_test_split(df, train_size=0.3, random_state= 42, stratify= df["target"])

In [5]:
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["target"])

In [6]:
x_train = train_df.drop(columns= ["target"])
y_train = train_df["target"]

In [7]:
x_val = val_df.drop(columns=["target"])
y_val = val_df["target"]

In [8]:
model = LogisticRegression(max_iter=5000)
model.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [9]:
val_pred = model.predict(x_val)

In [10]:
result = {
    "Accuracy" : accuracy_score(y_val, val_pred),
    "Precision" : precision_score(y_val, val_pred),
    "Recall" : recall_score(y_val, val_pred),
    "F1 score" : f1_score(y_val, val_pred)
}

In [11]:
print(pd.DataFrame([result]))

   Accuracy  Precision  Recall  F1 score
0  0.934673   0.911765   0.992  0.950192


In [12]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_val, val_pred)

cm_df = pd.DataFrame(cm,
                     index=["Actual Negative", "Actual Positive"],
                     columns=["Predicted Negative", "Predicted Positive"])

cm_df


,Predicted Negative,Predicted Positive
Actual Negative,62,12
Actual Positive,1,124


In [26]:
cm[0][1] / (cm[0][1] + cm[0][0])

0.16216216216216217

In [27]:
print(cm_df)

                 Predicted Negative  Predicted Positive
Actual Negative                  62                  12
Actual Positive                   1                 124


In [28]:
val_prob = model.predict_proba(x_val)[:, 1]

In [29]:
from sklearn.metrics import roc_auc_score, roc_curve
import plotly.express as px 
import plotly.graph_objects as go 

In [30]:
fpr, tpr, threshold = roc_curve(y_val, val_prob)
roc_auc = roc_auc_score(y_val,val_prob)

In [38]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = fpr, y= tpr, mode='lines',
    name=f"ROC Curve (AUC = {roc_auc:.3f})",
    line=dict(width=3)))
fig.add_trace(go.Scatter(
    x = [0,1],
    y = [0,1],
    mode="lines",
    line = dict(dash = "dash", color = "red"),
    name = "Random baseline",
    showlegend=True
))
fig.update_layout(xaxis_title = "False Positive Rate", yaxis_title = "Tru Positive rate (Recall)", width = 1100, height = 600)
fig.show()


In [32]:
from sklearn.metrics import precision_recall_curve, average_precision_score

In [33]:
precision , recall, threshold = precision_recall_curve(y_val, val_prob)
ap = average_precision_score(y_val, val_prob)

In [34]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x = recall, y = precision, mode = 'lines', 
    name = f"PR Curve (AP = {ap :.3f})",
    line = dict(width = 3)
))

fig.update_layout(
    title = "Precision-Recall Curve",
    xaxis_title = "Recall",
    yaxis_title = "precision",
    width = 600,
    height = 400
)

fig.show()
print("Average Precision (AP) : " , ap)

Average Precision (AP) :  0.9856499773088581


In [35]:
from sklearn.metrics import confusion_matrix 


def evaluate_threshold(threshold):
    preds = (val_prob >= threshold).astype(int)
    cm = confusion_matrix(y_val, preds)

    print(f"\n=== Threshold = {threshold} ===")
    print(pd.DataFrame(cm,
                       index = ["Actual Negative", "Actual Positivfe"], columns= ["Predicted Negative", "Predicted Positive"]))
    tpr = cm[1][1] / (cm[1][1] + cm[1][0])
    fpr = cm[0][1] / (cm[0][1] + cm[0][0])
    print(f"True + : {tpr}")
    print(f"False + : {fpr}")
    print("Precision :", precision_score(y_val, preds))
    print("Recall : ", recall_score(y_val, preds))
    print("F1 Score : ", f1_score(y_val, preds))


for th in [0.3,0.4,0.5,0.6,0.7]:
    evaluate_threshold(th)


=== Threshold = 0.3 ===
                  Predicted Negative  Predicted Positive
Actual Negative                   55                  19
Actual Positivfe                   1                 124
True + : 0.992
False + : 0.25675675675675674
Precision : 0.8671328671328671
Recall :  0.992
F1 Score :  0.9253731343283582

=== Threshold = 0.4 ===
                  Predicted Negative  Predicted Positive
Actual Negative                   60                  14
Actual Positivfe                   1                 124
True + : 0.992
False + : 0.1891891891891892
Precision : 0.8985507246376812
Recall :  0.992
F1 Score :  0.9429657794676806

=== Threshold = 0.5 ===
                  Predicted Negative  Predicted Positive
Actual Negative                   62                  12
Actual Positivfe                   1                 124
True + : 0.992
False + : 0.16216216216216217
Precision : 0.9117647058823529
Recall :  0.992
F1 Score :  0.9501915708812261

=== Threshold = 0.6 ===
                  P